## Let's learn OOP by practicing examples.

#### 0. A class definition for Circle

In [ ]:
import math

class Circle:
  nCircles = 0
  
  # 1) Use *args, **kargs for receiving any argument.
  # 2) super() returns a proxy object that is used to delegate method resolution 
  #    to the next class in the method resolution order(MRO).
  def __new__(cls, *args, **kargs): 
    print("Creating a new Circle instance")
    instance = super().__new__(cls)
    cls.nCircles += 1
    print(f"Number of Circle instances: {cls.nCircles}")
    return instance
  
  def __init__(self, x, y, r):
    self.x = x
    self.y = y
    self.r = r

  def __setattr__(self, name, value):
    print(f"Setting {name} to {value}")
    super().__setattr__(name, value)
    
  def area(self):
    return math.pi * self.r ** 2

#### 1. Creation

In [ ]:
crc1 = Circle(0, 1, 3)

In [ ]:
crc2 = Circle.__new__(Circle)
Circle.__init__(crc2, 1, 2, 3)

In [ ]:
crc3 = Circle.__new__(Circle)
Circle.__setattr__(crc3, 'x', 0)
Circle.__setattr__(crc3, 'y', 0)
Circle.__setattr__(crc3, 'r', 1)
Circle.area(crc3)

#### 2. Where are the attributes and methods of objects?  
 -    attributes of instance object ?&emsp; instance.\_\_dict__ 
 -    attributes and methods of class object ?&emsp;class.\_\_dict__

In [ ]:
# Let's create a circle instance, crc.
crc = Circle(10, 10, 10)

In [ ]:
# attributes of instance object lie in the instance.__dict__
print(crc.__dict__)

In [ ]:
# attributes and methods of class object lie in the Circle.__dict__.
print(Circle.__dict__)

In [ ]:
# Let's print the __dict__ as readable
print("{")
for k, v in Circle.__dict__.items():
    print(f"\t'{k}': \t\t{v}")
print("}")

#### 3. How to associate instance object 'crc' to class object 'Circle'?  
- instance.\_\_class__ refers to the class object
- Circle, class name, itself is a reference to the class object.
- In this example, crc.\_\_class__ == Circle.

In [ ]:
# instance.__class__ == class object
cls = crc.__class__
print(cls == Circle)

print('\n')
# If you want to look at the reference value, use id()
print(id(cls))
print(id(Circle))

print('\n')
# So, what are the types of cls and Circle?
print(cls)
print(Circle)

In [ ]:
a = 10

#### 5. How to find out the base (parent) class objects of an instance object?  
- className.\_\_bases__ : &emsp; Returns a tuple of base classes.

In [ ]:
# instance.__class__.__bases__ gives you a tuple of bases.
# Note that you might have multiple bases when you define a class like: class A(B, C).
print(crc.__class__.__bases__)

# To access the first base
print(crc.__class__.__bases__[0])

# Or, you can directly use the class name
print(Circle.__bases__)

In [ ]:
# Let's make a function that can traverse bases of a class.
# You should provide 'class reference' as an argument.
def traverse_bases(cls):
    while cls:
        yield cls
        if cls.__bases__ == ():
            break
        for p in cls.__bases__:
            cls = p

In [ ]:
# Let's go through our Circle.
for cls in traverse_bases(Circle):
    print(cls)
    
##### just putting some space for next print
print('\n')   

# You can pass the argument like crc.__class__
for cls in traverse_bases(crc.__class__):
    print(cls)

## Let's apply our knowledge about OOP to AI Programming!

#### 1. AIPParameter : Subclasses torch.Tensor for our AI programming.  
- AIPParameter overrides \_\_new(cls, data)__
- Inside the \_\_new__ method, it uses torch.Tensor.\_make_subclass(cls, data)

In [ ]:
import torch
class AIPParameter(torch.Tensor):
    def __new__(cls, data):
        if not isinstance(data, torch.Tensor):
            data = torch.tensor(data)
        param = torch.Tensor._make_subclass(cls, data)
        param.requires_grad = True
        return param

In [ ]:
# Let's create an instance of AIPParameter
param = AIPParameter([1., 2., 3.])
print(param)

In [ ]:
# Let's traverse the AIPParameter class object.
for cls in traverse_bases(AIPParameter):
    print(cls)
    
print('\n')

for cls in traverse_bases(param.__class__):
    print(cls)

#### 2. AIPModule : Overrides __setattr__(self, name, value)    
- \_\_setattr__ tracks AIPParameter and AIPModule instance objects in its own dictionaries
- Dictionaries: &emsp; \_parameters &emsp; and &emsp; \_modules 

In [ ]:
class AIPModule():
    def __init__(self):
        self._parameters = {}
        self._modules = {}
    def __setattr__(self, name, value):
        if isinstance(value, AIPParameter):
            self._parameters[name] = value
        elif isinstance(value, AIPModule):
            self._modules[name] = value
        super().__setattr__(name, value)
        
    def parameters(self):
        for param in self._parameters.values():
            yield param
        for module in self._modules.values():
            yield from module.parameters()

    def named_parameters(self, prefix=''):
        for name, param in self._parameters.items():
            yield prefix + name, param
        for module_name, module in self._modules.items():
            sub_prefix = f"{prefix}{module_name}."
            yield from module.named_parameters(prefix=sub_prefix)

In [ ]:
# Create an AIPModule
model = AIPModule()

# Add AIPParameters
model.param1 = AIPParameter([1., 2.])
model.param2 = AIPParameter(torch.randint(0, 10, [2, 2]).float())

# 1) Let's find out the parameters in instance object, model
print(model.__dict__)

# 2) Let's find out the parameters using instance method, parameters
print('\n')
for name, param in model.named_parameters():
    print(f"{name}:  {param}")


# 3) Let's traverse class object tree.
print('\n')
for cls in traverse_bases(model.__class__):
    print(cls)

#### 3. Subclassing AIPModule

In [ ]:
# A_Module subclasses AIPModule
class A_Module(AIPModule):
    def __init__(self):
        super().__init__()
        self.param2 = AIPParameter([4., 5, 6])
        self.param3 = AIPParameter([7., 8, 9])

# B_Module subclasses AIPModule and has a A_Module as an attribute.
class B_Module(AIPModule):
    def __init__(self):
        super().__init__()
        self.param1 = AIPParameter([1., 2, 3])
        self.AMoudle = A_Module()

In [ ]:
# Let's traverse A_Module
for cls in traverse_bases(A_Module):
    print(cls)

print('\n')

# Let's traverse B_Module
for cls in traverse_bases(B_Module):
    print(cls)

In [ ]:
# Let's findout the parameters in each module

# 1) A_Module
modelA = A_Module()
for name, param in modelA.named_parameters():
    print(f"{name}: {param}")

print('\n')

# 2) B_Module
modelB = B_Module()
for name, param in modelB.named_parameters():
    print(f"{name}: {param}")

In [ ]:
# Let's compare Python __dict__
for k, v in modelB.__dict__.items():
    print(f"{k}:  {v}")

#### 4. AIPLinear Model
- AIPLinear(in_features, out_features, bias=True)
- weight :&emsp;[out_features, in_features]
- biase : &emsp; [out_featuers]

In [ ]:
class AIPLinear(AIPModule):
    def __init__(self, in_features, out_features, use_bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.use_bias = use_bias
        
        self.weight = AIPParameter(torch.randn(out_features, in_features) * (1.0 / in_features**0.5))
        
        if use_bias:
            self.bias = AIPParameter(torch.zeros(out_features))
        else:
            self.bias = None
    
    def __call__(self, x):
        output = x @ self.weight.T 
        
        if self.use_bias and self.bias is not None: 
            output = output + self.bias
        
        return output

In [ ]:
aiplinear = AIPLinear(4, 2)

In [ ]:
for cls in traverse_bases(AIPLinear):
    print(cls)

In [ ]:
for name, param in aiplinear.parameters():
    print(f"{name}: {param}")

#### 5. MLP (Multi-Linear Perceptron)

In [ ]:
def relu(x):
    return torch.maximum(torch.tensor(0.0), x)

In [ ]:
class MLP(AIPModule):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = AIPLinear(input_dim, hidden_dim)
        self.fc2 = AIPLinear(hidden_dim, output_dim)

    def __call__(self, x):
        x = relu(self.fc1(x))
        x = self.fc2(x)
        return x
    
mlp = MLP(2, 4, 1)

In [ ]:
for cls in traverse_bases(MLP):
    print(cls)

In [ ]:
for name, param in mlp.named_parameters():
    print(f"{name}: {param}")